In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import torch
from torch import nn
import keras
from keras.layers import TorchModuleWrapper
from torch.nn import functional as F
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Define the dimensions based on the model
batch_size = 64
input_features = 32
num_classes = 10
# 1. Generate random input data (x)
# Shape: (batch_size, input_features) -> e.g., (64, 32)
# Using float32 is standard for deep learning frameworks.
x_test = np.random.rand(batch_size, input_features).astype('float32')

# 2. Generate random integer labels (y)
# Shape: (batch_size,) -> e.g., (64,)
# The labels are integers from 0 to 9 for 10 classes.
y_test = np.random.randint(0, num_classes, size=batch_size)

print("Shape of x_test:", x_test.shape)
print("Shape of y_test:", y_test.shape)
print("Sample labels:", y_test[:5])

Shape of x_test: (64, 32)
Shape of y_test: (64,)
Sample labels: [8 1 9 2 8]


In [4]:
class PyTorchModelWithBatchNorm(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.dense1 = nn.Linear(input_features, 64)
        self.bn1 = nn.BatchNorm1d(64) # BatchNorm layer
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(64, output_features)

    # CRITICAL: The forward method must accept a 'training' argument.
    def forward(self, x):
        # Use the flag to set the mode for BatchNorm/Dropout
        
        self.eval()

        x = self.dense1(x)
        x = self.bn1(x) # BatchNorm will now behave correctly
        x = self.relu(x)
        x = self.dense2(x)
        return x

In [5]:
# Instantiate the PyTorch model
pytorch_model = PyTorchModelWithBatchNorm(input_features=32, output_features=10)
pytorch_model.eval()

PyTorchModelWithBatchNorm(
  (dense1): Linear(in_features=32, out_features=64, bias=True)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (dense2): Linear(in_features=64, out_features=10, bias=True)
)

In [6]:
pytorch_model(torch.from_numpy(x_test[:1]))

tensor([[-0.1362, -0.0726, -0.0211, -0.1948,  0.0368, -0.0592, -0.1668, -0.1452,
          0.0440, -0.1805]], grad_fn=<AddmmBackward0>)

In [7]:
# Wrap it in a single step
wrapped_model_block = TorchModuleWrapper(pytorch_model)

# Build the final Keras model
model = keras.Sequential([
    keras.layers.Input(shape=(32,)),
    wrapped_model_block
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ torch_module_wrapper            │ (None, 10)             │         2,890 │
│ (TorchModuleWrapper)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,890 (11.29 KB)

 Trainable params: 2,890 (11.29 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
pytorch_model(torch.from_numpy(x_test[:1]).to(device))

tensor([[-0.1362, -0.0726, -0.0211, -0.1948,  0.0368, -0.0592, -0.1668, -0.1452,
          0.0440, -0.1805]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [9]:
model.predict(x_test[:1])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


array([[-0.13616914, -0.07261349, -0.02107635, -0.19483872,  0.03682116,
        -0.05917181, -0.1668364 , -0.14515658,  0.0440481 , -0.1805081 ]],
      dtype=float32)

In [10]:
model.train()

<Sequential name=sequential, built=True>

In [11]:
input_tensor = torch.from_numpy(x_test[:1]).to(device)
input_tensor.requires_grad_(True)
# 2. Effectuez la passe avant (forward pass)
output = pytorch_model(input_tensor)

# 3. Calculez les gradients (backward pass)
# .backward() ne fonctionne que sur un scalaire.
# On somme donc la sortie pour en faire un scalaire.
# Cela calcule le gradient de la somme des sorties par rapport à l'entrée.
output.sum().backward()

In [12]:
gradient = input_tensor.grad
print("Gradient :")
print(gradient)


Gradient :
tensor([[ 0.0193,  0.1992, -0.1118,  0.0701, -0.0501,  0.0773, -0.1161, -0.2728,
         -0.4056, -0.0447, -0.1154, -0.0285,  0.1527,  0.0350,  0.1253,  0.0548,
          0.1969, -0.0081, -0.0365, -0.1880, -0.0418, -0.3044, -0.0455, -0.1224,
         -0.0116,  0.0072, -0.1729, -0.2041, -0.0487, -0.0957, -0.1588, -0.1244]],
       device='cuda:0')


In [13]:
input_tensor = torch.from_numpy(x_test[:1]).to(device)
input_tensor.requires_grad_(True)
# 2. Effectuez la passe avant (forward pass)
output = model(input_tensor)

# 3. Calculez les gradients (backward pass)
# .backward() ne fonctionne que sur un scalaire.
# On somme donc la sortie pour en faire un scalaire.
# Cela calcule le gradient de la somme des sorties par rapport à l'entrée.
output.sum().backward()

In [14]:
gradient = input_tensor.grad
print("Gradient :")
print(gradient)


Gradient :
tensor([[ 0.0193,  0.1992, -0.1118,  0.0701, -0.0501,  0.0773, -0.1161, -0.2728,
         -0.4056, -0.0447, -0.1154, -0.0285,  0.1527,  0.0350,  0.1253,  0.0548,
          0.1969, -0.0081, -0.0365, -0.1880, -0.0418, -0.3044, -0.0455, -0.1224,
         -0.0116,  0.0072, -0.1729, -0.2041, -0.0487, -0.0957, -0.1588, -0.1244]],
       device='cuda:0')


In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ torch_module_wrapper            │ (None, 10)             │         2,890 │
│ (TorchModuleWrapper)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,890 (11.29 KB)

 Trainable params: 2,890 (11.29 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# This is our custom adapter module
class PretrainedModelWrapper(nn.Module):
    def __init__(self, pretrained_model):
        super().__init__()
        # Load the pre-trained model inside the wrapper
        self.pretrained_model = pretrained_model
    # Implement the Keras-compatible forward method
    def forward(self, x):
        # 1. Use the training flag to set the mode
        self.pretrained_model.eval()

        # 2. Call the original model's forward pass
        return self.pretrained_model(x)